# Exercise 10- Stacking

In this exercise you will implement an ensemble method by learning a stacked regressor.

In the event of a persistent problem, do not hesitate to contact the course instructor under

- maurice.wenig@uni-jena.de

### Submission
- Deadline of submission: 22.06.2026 23:59
- Submission on [moodle page](https://moodle.uni-jena.de/course/view.php?id=76636)


# The Dataset

We will use a real world dataset used for predicting the [quality of red wine](https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009).
Altough the quality is a discrete value between 0 and 10, we interpret it as a regression task. 

### Task 1

Load the dataset stored in `dataset.csv` and split it into `x` and `y`.

In [1]:
# TODO: load data
import numpy as np
import polars as pl


data = pl.read_csv("dataset.csv")
y = data.get_column("quality").to_numpy()
x = data.drop("quality").to_numpy()


# assertions
assert x.shape == (1599, 11)
assert y.shape == (1599,)

## $R^2$ Score

Sklearn uses the [$R^2$ score](https://en.wikipedia.org/wiki/Coefficient_of_determination) as a quality measure for regressors. Given true values $y$ and predicted values $\hat{y}$ the $R^2$ score is defined as 

\begin{align*}
R^2(y, \hat{y}) &= 1-\cfrac{\sum_{i=1}^m(y_i-\hat{y}_i)^2}{\sum_{i=1}^m(y_i - \bar{y})^2}\,,
\end{align*}
where $\bar{y}$ is the average of $y$.

This value is 1 if the predictions match exactly, 0 if we would simply always predict the average and negative if our predictions are worse than this simple baseline.\
In short we aim for a value $>0$ and close to $1$.

### Task 2

Implement the $R^2$ score.\
Then use scikit learns [Linear Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) model to fit on the dataset and calculate the $R^2$ score.\
Compare your result to the `.score` method of the regressor.

In [2]:
import numpy.typing as npt
from sklearn.linear_model import LinearRegression


def r2_score(y: npt.NDArray[np.float64], y_hat: npt.NDArray[np.float64]) -> float:
    """Computes the coefficient of determination.

    Parameters
    ----------
    y : npt.NDArray[np.float64]
        True labels, shape (n_samples,).
    y_hat : npt.NDArray[np.float64]
        Predicted labels, shape (n_samples,).

    Returns
    -------
    float
        Score in the range (-inf, 1).
    """

    # TODO: implement
    return  1 - np.sum((y - y_hat) ** 2) / np.sum((y - np.mean(y)) ** 2)


# TODO: calculate r2 score for linear regressor, compare with .score
regressor = LinearRegression().fit(x, y)
y_hat = regressor.predict(x)
my_score = r2_score(y, y_hat)
sklearn_score = regressor.score(x, y)

print("my_score:", my_score)
print("sklearn_score:", sklearn_score)
assert np.isclose(my_score, sklearn_score)

# assertions
np.random.seed(0)
p = np.random.rand(100)
p_hat = 2 * p + np.random.rand(100) * 0.001
assert np.isclose(r2_score(p, p_hat), -2.695248533930105)

my_score: 0.36055170303868833
sklearn_score: 0.36055170303868833


# Stacking

The main idea in stacking is to 
1. learn several heterogenous base models on the original data
2. learn a meta model on the predictions of the base models

<div>
<img src="images/stacking.png" width="600"/>
</div>
The hope is that the meta model can learn to combine the strengths of the base models (e.g. if model 1 fails, model 3 is strong).
Note that in contrast to bagging and boosting the base models must not be of the same method (e.g. decision trees).

## Base Models

First lets select a set of base models. We can now choose from the wide pool of regression methods.

Here we want to use the following models:
- [Linear Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) (with NO extra keywords)
- Polynomial Regression of degree 2 (use a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html) of [Polynomial Features](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) followed by Linear Regression) (with NO extra keywords)
- [KNN Regression](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html) (with `n_neighbours=10`)
- [Decision Tree Regression](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html) (with `max_depth=4` and `random_state=42`)


### Task 3
Create a list of base models and evaluate them using crossvalidation (avg. over 10 folds).

In [3]:
from sklearn.base import BaseEstimator
from sklearn.model_selection import cross_val_score

from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor

# TODO: create base models
base_models: list[tuple[str, BaseEstimator]] = [
    ("linear", LinearRegression()),
    ("knn", KNeighborsRegressor()),
    ("poly", make_pipeline(PolynomialFeatures(degree=2), LinearRegression())),
    ("tree", DecisionTreeRegressor()),
]

# TODO: estimate avg. crossvalidation score for each base model
cross_val_means = []
for name, model in base_models:
    scores = cross_val_score(model, x, y, cv=5)
    mean_score = np.mean(scores)
    cross_val_means.append(mean_score)
    print(f"{name}: {mean_score:.4f}")

linear: 0.2900
knn: -0.0484
poly: 0.2301
tree: -0.3377


## Meta Model

The meta model uses the predictions of the base models to predict $y$. One can thus view the base models as a feature map for the meta model.

In order to train the meta model, **we need the predictions of the base models on unseen data** since this is the scenario we would face at inference time. A simple method is to use **out-of-fold predictions** during training:

1. separate the data into k folds (deterministically - you don't have to shuffle before).
2. hold out one of the folds and train the base models on the other folds.
3. predict the held out fold using the base models.
4. repeat the above two steps k times to obtain out-of-fold predictions for all k folds.
5. feed all the out-of-fold prediction as features (training data) to the meta model.


### Task 4

Implement the out-of-fold method below.\
Calculate the $R^2$-Score on the out-of-fold predictions for each of the base models.

In [4]:
def oof_prediction(model: BaseEstimator, x: npt.NDArray[np.float64], y: npt.NDArray[np.float64], n_folds: int = 5) -> npt.NDArray[np.float64]:
    """Computes out-of-fold predictions.

    Parameters
    ----------
    model : BaseEstimator
        Model with .fit and .predict methods.
    x : npt.NDArray[np.float64]
        Features, shape (n_samples, n_features).
    y : npt.NDArray[np.float64]
        True labels, shape (n_samples,).
    n_folds : int, optional
        Amount of folds, by default 5.

    Returns
    -------
    npt.NDArray[np.float64]
        Predictions for each sample, such that the model was trained on all folds except the one, which the sample belongs to. Shape (n_samples,).
    """

    # TODO: implement fold splitting
    folds_indices = np.array_split(np.arange(x.shape[0]), n_folds)
    assert np.all([len(fold_indices) >= x.shape[0] // n_folds for fold_indices in folds_indices]), "Some folds are too small."
    assert np.all([len(fold_indices) <= x.shape[0] // n_folds + 1 for fold_indices in folds_indices]), "Some folds are too big."
    assert np.sum([len(fold_indices) for fold_indices in folds_indices]) == x.shape[0], "Either too little or too many indices."
    assert np.all(np.concatenate(folds_indices) == np.arange(x.shape[0])), "Fold indices were permuted."
    # TODO: implement out-of-fold predictions
    oof_preds = np.empty(x.shape[0], dtype=np.float64)
    for fold_indices in folds_indices:
        train_indices = np.setdiff1d(np.arange(x.shape[0]), fold_indices)
        model.fit(x[train_indices], y[train_indices])
        oof_preds[fold_indices] = model.predict(x[fold_indices])
    return oof_preds


# TODO: calculate r2 score for oof predictions for each base model
r2_scores = []
for name, model in base_models:
    oof_preds = oof_prediction(model, x, y)
    score = r2_score(y, oof_preds)
    r2_scores.append(score)
    print(f"{name}: {score:.4f}")


linear: 0.3301
knn: 0.0006
poly: 0.2789
tree: -0.3059


Now lets put everything together.

### Task 5

Implement the following `Stacking` class. Keep in mind the following things:
- the meta model is trained on out-of-fold predictions of the base models
- the base models are trained on the given dataset
- when predicting, we just use the predictions of the base models (no out-of-fold) as input for the meta model

Use your class to learn a stacked regressor with **linear regression as meta model** and the base models from Task 3. Evaluate it using crossvalidation (avg. of 10 folds) and compare the score to those of the base models (Task 3).

In [5]:
from typing import Self


class StackedRegressor(BaseEstimator):
    def __init__(self, base_models: list[tuple[str, BaseEstimator]], meta_model: BaseEstimator, n_folds: int = 5):
        self.base_models = base_models
        self.meta_model = meta_model
        # n folds used for oof_prediction during training
        self.n_folds = n_folds

    def fit(self, x: npt.NDArray[np.float64], y: npt.NDArray[np.float64]) -> Self:
        """Learns base models and meta model.

        Parameters
        ----------
        x : npt.NDArray[np.float64]
            Features, shape (n_samples, n_features).
        y : npt.NDArray[np.float64]
            True labels, shape (n_samples,).
        """

        # TODO: implement
        x_meta = np.column_stack([
            oof_prediction(model, x, y, self.n_folds) for _, model in self.base_models
        ])
        for _, model in self.base_models:
            model.fit(x, y)
        self.meta_model.fit(x_meta, y)
        return self

    def predict(self, x: npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
        """Given the features, predict the labels with the base classes, and then predict the label based on the base predictions.

        Parameters
        ----------
        x : npt.NDArray[np.float64]
            Features, shape (n_samples, n_features).

        Returns
        -------
        npt.NDArray[np.float64]
            Predicted label for each sample, shape (n_samples,).
        """

        # TODO: implement
        x_meta = np.column_stack([model.predict(x) for _, model in self.base_models])
        return self.meta_model.predict(x_meta)

    def score(self, x: npt.NDArray[np.float64], y: npt.NDArray[np.float64]) -> float:
        """Computes the coefficient of determination.

        Parameters
        ----------
        x : npt.NDArray[np.float64]
            Features, shape (n_samples, n_features).
        y : npt.NDArray[np.float64]
            True labels, shape (n_samples,).

        Returns
        -------
        float
            Score in the range (-inf, 1).
        """

        # TODO: implement
        y_hat = self.predict(x)
        return r2_score(y, y_hat)


# TODO: fit stacked model
meta_model = LinearRegression()
stacked_model = StackedRegressor(base_models, meta_model, n_folds=5)
stacked_model.fit(x, y)

# TODO: evaluate with crossvalidation, compare to base models
cross_val_mean = cross_val_score(stacked_model, x, y, cv=10).mean()
print(f"stacked_model: {cross_val_mean:.4f}")

stacked_model: 0.2474


### Task 6

Use the [scikit-learn implementation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingRegressor.html) to learn a stacked regressor.
Evaluate it using crossvalidation (avg. of 10 folds) and compare the score to the scores of task 5.

Note that minor differences can occur due to a more advanced oof-prediction used in sklearn.

In [6]:
from sklearn.ensemble import StackingRegressor

# TODO: fit sklearn stacked model
sklearn_stacked_model = StackingRegressor(estimators=base_models, final_estimator=meta_model, cv=5)
sklearn_stacked_model.fit(x, y)

# TODO: evaluate with crossvalidation, compare to custom model
cv_mean_sklearn = cross_val_score(sklearn_stacked_model, x, y, cv=10).mean()
print(f"sklearn_stacked_model: {cv_mean_sklearn:.4f}")
print(f"custom stacked_model: {cross_val_mean:.4f}")

sklearn_stacked_model: 0.2460
custom stacked_model: 0.2474


### Task 7
Try at least two different combinations of regressors for base models and meta model and report the average crossvalidation score.
[Here](https://scikit-learn.org/stable/supervised_learning.html) you can find an overview page of sklearn estimators.

In [7]:
# TODO: try different combinations
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Combination 1: strong non-linear base models + linear meta model
combo1_base = [
    ("rf", RandomForestRegressor(n_estimators=100, random_state=42)),
    ("gbr", GradientBoostingRegressor(random_state=42)),
    ("knn", KNeighborsRegressor(n_neighbors=10)),
]
combo1_meta = Ridge()
combo1 = StackedRegressor(combo1_base, combo1_meta, n_folds=5)
combo1_score = cross_val_score(combo1, x, y, cv=10).mean()
print(f"combo1 (RF + GBR + KNN -> Ridge): {combo1_score:.4f}")

# Combination 2: linear/poly/tree base models + decision tree meta model
combo2_base = [
    ("poly", make_pipeline(PolynomialFeatures(degree=2), LinearRegression())),
    ("tree", DecisionTreeRegressor(max_depth=4, random_state=42)),
    ("linear", LinearRegression()),
]
combo2_meta = DecisionTreeRegressor(max_depth=3, random_state=42)
combo2 = StackedRegressor(combo2_base, combo2_meta, n_folds=5)
combo2_score = cross_val_score(combo2, x, y, cv=10).mean()
print(f"combo2 (poly + tree + linear -> tree): {combo2_score:.4f}")


combo1 (RF + GBR + KNN -> Ridge): 0.2776
combo2 (poly + tree + linear -> tree): 0.2019
